1. Definição do problema
Você pode apresentar assim:

* A Quinav-Distribuidora inaugurou o setor de e-commerce.

* O desafio é entender se o canal está performando bem.

* O foco é descobrir pontos de melhoria em vendas, pedidos, clientes, produtos e logística.

Etapa 1: Importar as bibliotecas e Criar a Base de Dados.

In [498]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import locale
from pathlib import Path

#locale.setlocale(locale.LC_TIME, 'pt_BR.UTF-8')

Etapa 2: Definir configurações iniciais.

In [499]:
# Garante que os dados sejam sempre os mesmo.
np.random.seed(42)

# Definir um estilo visual padrão para os gráficos,
# com grade branca ao fundo
sns.set_theme(style='whitegrid')

# Cria um Objeto representando a pasta onde os arquivos,
# serão salvos.
out = Path('output')

# Cria a pasta output se ainda não existir
out.mkdir(exist_ok=True)

Etapa 3:Definir o tamanho da base.

In [500]:
numero_linhas = 3000

Etapa 4: Criar a base de dados.

In [501]:
# Cria um data frame com os dados
base_dados = pd.DataFrame({
    'id_pedido':[f'ORD{i:06d}' for i in range(1, numero_linhas + 1)], # Cria a coluna 'id_pedido', com ORD1 -ORD3000
    
    'data':pd.to_datetime(
        np.random.choice(pd.date_range('2025-01-01','2025-12-31', 
        freq='D'),
        numero_linhas) # Cria a coluna data
    ),

    'id_cliente':np.random.randint(1000, 5000, numero_linhas),

    'tipo_cliente':np.random.choice(
        ['Novo','Recorrente'], numero_linhas, p=[0.55, 0.45]
        ),  # Cria a coluna tipo do cliente, set percentual de 55% para novo e 45% recorrente
    
    'estado':np.random.choice(
        ['SP','RJ','MG','ES','PR','SC','RS','BA','GO','PE'],
        numero_linhas),

    'regiao': np.random.choice(
        ['Sudeste','Sul','Nordeste', 'Centro-Oeste'],
        numero_linhas),

    'id_produto': np.random.randint(10000, 99999, numero_linhas),

    'categoria_Produto': np.random.choice(
        ['Limpeza', 'Comida', 'Bebidas', 'Escritórios',
        'Papelaria','Equipamento Industrial'],numero_linhas),

    'quantidade': np.random.randint(1, 15, numero_linhas),

    'preco_unitario': np.round(
        np.random.uniform(10, 500, numero_linhas),
        2),
    
    'modo_pagamento': np.random.choice(
        ['Pix', 'Cartão de Crédito', 'Cartão de Debito', 'Boleto Bancário'],
        numero_linhas
        ),
    
    'canal': np.random.choice(
        ['Google Ads', 'Instagram','WhatsApp','Orgânico','Marketplace'],
        numero_linhas
        ),

    'status': np.random.choice(
        ['Entregue', 'Entregue com atraso', 'Cancelado', 'Devolvido'],
        numero_linhas,
        p=[0.5, 0.3, 0.1, 0.1]
        ),
    
    'tempo_entrega_dias': np.random.randint(1, 15, numero_linhas),

    'avaliacao': np.random.randint(1, 6, numero_linhas),
    
    'cupom_usado' : np.random.choice([0, 1], numero_linhas, p=[0.7, 0.3])
})

df_analise = base_dados

Etapa 4.1: visualizando as informações da base de dados (Opicional).

In [502]:
#display(base_dados.info())
#display(base_dados.head())
#display(base_dados.describe())

Etapa 5: Criar colunas calculadas

In [503]:
df_analise['valor_bruto'] = np.round(df_analise['quantidade'] * df_analise['preco_unitario'],2)
df_analise['custo_frete'] =np.round(np.random.uniform(8, 60, numero_linhas), 2)
df_analise['valor_desconto'] = np.round(
    np.where(
        df_analise['cupom_usado']==1,
        df_analise['valor_bruto'] * 0.1, 0)
        ,2)
df_analise['valor_final'] = (df_analise['valor_bruto'] - 
                             df_analise['valor_desconto'] + 
                             df_analise['custo_frete'])

display(df_analise.head())

,id_pedido,data,id_cliente,tipo_cliente,estado,regiao,id_produto,categoria_Produto,quantidade,preco_unitario,modo_pagamento,canal,status,tempo_entrega_dias,avaliacao,cupom_usado,valor_bruto,custo_frete,valor_desconto,valor_final
0,ORD000001,2025-04-13,2930,Novo,GO,Centro-Oeste,31346,Bebidas,11,441.09,Boleto Bancário,WhatsApp,Entregue,11,3,0,4851.99,50.63,0.0,4902.62
1,ORD000002,2025-12-15,3861,Recorrente,BA,Centro-Oeste,20619,Bebidas,7,225.75,Cartão de Crédito,Instagram,Entregue com atraso,14,3,0,1580.25,46.99,0.0,1627.24
2,ORD000003,2025-09-28,4958,Recorrente,RJ,Nordeste,11766,Papelaria,4,316.35,Cartão de Debito,Google Ads,Entregue,4,4,0,1265.40,59.59,0.0,1324.99
3,ORD000004,2025-04-17,4087,Novo,BA,Sudeste,13549,Equipamento Industrial,5,250.30,Pix,Instagram,Entregue com atraso,8,1,0,1251.50,14.92,0.0,1266.42
4,ORD000005,2025-03-13,2152,Recorrente,ES,Centro-Oeste,28693,Equipamento Industrial,11,437.56,Boleto Bancário,Google Ads,Entregue,9,3,0,4813.16,45.20,0.0,4858.36


Etapa 6: Criar colunas de tempo

In [504]:
df_analise['ano']  = df_analise['data'].dt.year
df_analise['mes']  = df_analise['data'].dt.month
df_analise['dia_semana'] = df_analise['data'].dt.day_name(locale='Portuguese_Brazil.1252')

Etapa 7: Ajustar a satisfação

In [505]:
df_analise.loc[df_analise['status'] != 'Entregue','avaliacao'] = np.nan

Etapa 8: Salvar a base em CSV

In [506]:
df_analise.to_csv(out / 'quinav_ecommerce_data.csv',index=False)

Etapa 9: Gráfico de receita por categoria

In [507]:
# Cria um agrupamento de receito por categoria de produto
categoria = (
    df_analise.groupby(
    'categoria_Produto',
    as_index=False)
    ['valor_final']
    .sum()
    .sort_values('valor_final', ascending=False)
)
# Cria o gráfico com a agrupamento de categoria
plt.figure(figsize=(10, 8))

ax = sns.barplot(
        data=categoria,
        x='categoria_Produto',
        y='valor_final',
        palette='Blues_r'
        )

# Ajustando as legendas do gráfico
plt.title('Receita por Categoria',
          fontsize=17,
          fontweight='bold')
plt.xlabel('Categoria', fontsize=13)
plt.ylabel('Receita Total (R$)',fontsize=13)

# Adicionando rótulos de dados para os valores
for i, valor in enumerate(categoria['valor_final']):
    ax.text(
        x=i,                        # posição horizontal (índice da barra)
        y=valor + (valor * 0.01),     # posição vertical (um pouco acima da barra)
        s=f'R$ {valor:,.0f}',       # texto formatado com R$ e separador de milhar
        ha='center',                # centralizado
        va='bottom',                # alinhado embaixo
        fontsize=10,
        fontweight='bold'
    )


# Ajustando a proporção do gráfico
plt.ylim(bottom=0)
plt.subplots_adjust(bottom=0)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(out/'receita_por_categoria.png', dpi=200, bbox_inches='tight')
plt.close()


C:\Users\SANTIAGO\AppData\Local\Temp\ipykernel_1276\1460150329.py:13: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.barplot(


Etapa 10: Gráfico de status dos pedidos

In [508]:
status = df_analise['status'].value_counts().reset_index()
status.columns =['status','contagem']
plt.figure(figsize=(8, 5))

ax=sns.barplot(
    data=status,
    x='status',
    y='contagem',
    palette='Set2'
)
plt.title('Status de Distribuição')
plt.xlabel('Status')
plt.ylabel('Contagem'),

# Colocando rótulo de dados acima
for i, valor in enumerate(status['contagem']):
    ax.text(
        x = i,                          # posição horizontal (índice da barra)
        y= valor + (valor * 0.005),     # posição vertical (um pouco acima da barra)
        s= f'{valor:,.0f}',             # texto formatado com separador de milhar
        ha = 'center',                  # centralizado
        va = 'bottom',                  # alinhado embaixo
        fontsize=10,
        fontweight = 'bold'
    )

plt.ylim(bottom=0)
plt.subplots_adjust(bottom=0)
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig(out/'status_distribuição.png',dpi=200, bbox_inches='tight')
plt.close()



C:\Users\SANTIAGO\AppData\Local\Temp\ipykernel_1276\803123130.py:5: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax=sns.barplot(


Etapa 11: Gráfico de receita por região

In [509]:
# Criando 
regiao = (df_analise
            .groupby('regiao', as_index=False)
            ['valor_final']
            .sum()
            .sort_values('valor_final', ascending=False)
            )

plt.figure(figsize=(8, 5))
 
# Alimentando o gráfico com os dados e escolhendo a paleta de cores
ax = sns.barplot(
    data=regiao,
    x='regiao',
    y='valor_final',
    palette='Greens_r'
)

# Colocando rótulo de dados acima
for i, valor in enumerate(regiao['valor_final']):
    ax.text(
        x = i,
        y = valor + (valor * 0.005),
        s = f'R$ {valor:,.0f}',
        ha = 'center',
        va = 'bottom',
        fontsize = 10,
        fontweight = 'bold'
    )

plt.ylim(bottom=0)
plt.subplots_adjust(bottom=0)
plt.title('Receita por Região')
plt.xlabel('regiao')
plt.ylabel('receita')
plt.tight_layout()
plt.savefig(out / 'receita_por_regiao.png', dpi=200, bbox_inches='tight')
plt.close()

C:\Users\SANTIAGO\AppData\Local\Temp\ipykernel_1276\2212395609.py:12: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.barplot(


Etapa 12: Gráfico de satisfação x entrega

In [510]:
# Criado valores mais reais para avaliação
# 1.Definido penalidade base por dia de entrega
penalidade_por_dia = df_analise['tempo_entrega_dias'] * 0.2 # - 2 pontos fixados por dia

# 2.Definido a penalidade fixa por atraso
# Se status for 'Entregue com atraso', subtraímos 2, caso contrário, 0.
penalidade_status = np.where(df_analise['status']=='Entregue com atraso',2,0)

# 3.Aplicamos a lógica completa:
base_avaliacao = (
    5.5 - penalidade_por_dia -
    penalidade_status + 
    np.random.normal(0,0.2,numero_linhas)
    )

df_analise['avaliacao'] = np.where(
    df_analise['status'].isin(['Entregue', 'Entregue com atraso']),
    base_avaliacao.clip(1, 5).astype(int),
    np.nan
)


df_satisfacao_entregas = df_analise.dropna(subset=['avaliacao','tempo_entegra_dias'])

plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=df_satisfacao_entregas,
    x='tempo_entregra_dias',
    y='avaliacao',
    hue='status',
    alpha=0.4
)
plt.ylim(bottom=0)
plt.subplots_adjust(bottom=0)
plt.title('Avaliação do Cliente vs Tempo de Entrega')
plt.xlabel('tempo_entregra_dias')
plt.ylabel('avaliacao')
plt.legend(title='Status', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.savefig(out/'avaliacao_vs_tempo_entegra_dias.png', dpi=200, bbox_inches='tight')
plt.close()

# Validando se há correlação entre as variáveis
df_satisfacao_entregas_correlacao = df_satisfacao_entregas[['avaliacao','tempo_entegra_dias']]
df_satisfacao_entregas_correlacao.corr()

KeyError: ['tempo_entegra_dias']

Etapa 13: Gráfico de receita mensal

In [ ]:
df_receita_mensal = (
    df_analise.assign(
       ano_mes=df_analise['data']
        .dt.to_period('M')
        .astype(str))
        .groupby('ano_mes', as_index=False)
        ['valor_final']
        .sum())

plt.figure(figsize=(12, 5))

sns.lineplot(
    data=df_receita_mensal,
    x='ano_mes',
    y='valor_final',
    marker='o',
    color='#2a6fdb'
    )

plt.ylim(bottom=0)
plt.subplots_adjust(bottom=0)
plt.title('Receita Mensal')
plt.xlabel('Mes')
plt.ylabel('Receita')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(out/'receita_mensal.png',dpi=200, bbox_inches='tight')
plt.close()

Etapa 14: KPIS de negócio

In [ ]:
kpis = pd.DataFrame({
    'metrica':['Faturamento Total',
                'Ticket Médio',
                'Taxa Cancelamento',
                'Avaliação Média'],
    'valor' : [
        f'R$ {df_analise["valor_final"].sum():,.2f}',
        f'R$ {df_analise["valor_final"].mean():,.2f}',
        f'{df_analise["status"].eq("Cancelado").mean():.1%}',
        f'{df_analise["avaliacao"].mean():.1f}/5'
    ]
})

display(kpis)

,metrica,valor
0,Faturamento Total,"R$ 5,770,153.06"
1,Ticket Médio,"R$ 1,923.38"
2,Taxa Cancelamento,9.5%
3,Avaliação Média,4.2/5


Etapa 15: Insights de negócio

## Principais insights:

1. **Equipamento Industrial lidera faturamento** (X% do total)
2. **Sudeste concentra 65% da receita** — priorizar logística
3. **10% cancelamentos** — investigar motivos (frete alto?)
4. **30% usam cupom** — aumentar ofertas para novos clientes
5. **Avaliação média 3.2/5** — foco em experiência de entrega